
# System Handler – AI Study Assistant + Ollama Local

## Notebook oficial – OpenAI + Anthropic Claude + Ollama

Este notebook toma como referencia el notebook `system-handler-ai-study-assistant`,
pero aquí nos enfocamos en una extensión nueva:

- conectar el asistente a **OpenAI**
- conectar el asistente a **Anthropic Claude**
- conectar el asistente a **Ollama local**
- reforzar el concepto de **contexto**
- tener un **Modo Quiz** con corrección local y botón de **Reintentar incorrectas**

> El código base que ya habíamos explicado antes se concentra aquí en una sola sección.
> Si quieres entender ese bloque a detalle, revisa el notebook original.



## Celda 1 — Instalar dependencias

Usaremos:

- `openai`
- `anthropic`
- `gradio`
- `ipywidgets`
- `requests`

Si no estas usando `uv` puedes instalar dependencias con pip.

```
%pip install -q openai anthropic gradio ipywidgets requests
```


## Celda 2 — Cómo instalar Ollama

**Ollama** te permite correr modelos en tu propia computadora.
Eso lo vuelve muy interesante porque puedes practicar de forma local y, en muchos casos,
**sin pagar por cada request**.

### macOS
1. Descarga e instala Ollama.
2. Abre la app o verifica que el servicio esté corriendo.
3. Descarga un modelo, por ejemplo:

```bash
ollama pull llama3.2
```

4. Prueba que responde:

```bash
ollama run llama3.2
```

### URL local por defecto
Ollama normalmente expone su API aquí:

```text
http://localhost:11434
```



## Celda 3 — Importaciones

Nada raro aquí. Solo cargamos lo necesario para crear el asistente y conectar varios proveedores.


In [1]:

import os
import re
import json
import requests
from typing import Any, Dict, List, Optional

import gradio as gr
from openai import OpenAI
from anthropic import Anthropic



## Celda 4 — Configuración de credenciales

Ahora este notebook puede trabajar con tres rutas:

- **OpenAI** → requiere `OPENAI_API_KEY`
- **Anthropic Claude** → requiere `ANTHROPIC_API_KEY`
- **Ollama local** → no requiere API key si corre en `http://localhost:11434`

> Si quieres entender más del flujo base del asistente, revisa el notebook original `system-handler-ai-study-assistant`.


In [2]:

# Si quieres usar OpenAI, configura tu key aquí o como variable de entorno.
# os.environ["OPENAI_API_KEY"] = "sk-..."

# Si quieres usar Anthropic Claude, configura tu key aquí o como variable de entorno.
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

print("OPENAI_API_KEY detectada:" if OPENAI_API_KEY else "OPENAI_API_KEY no detectada.")
print("ANTHROPIC_API_KEY detectada:" if ANTHROPIC_API_KEY else "ANTHROPIC_API_KEY no detectada.")
print(f"OLLAMA_BASE_URL: {OLLAMA_BASE_URL}")


OPENAI_API_KEY detectada:
ANTHROPIC_API_KEY detectada:
OLLAMA_BASE_URL: http://localhost:11434



## Celda 5 — Código reutilizado + extensión del asistente

Esta celda concentra el bloque principal que ya habíamos trabajado en el notebook anterior:

- prompts base
- control de contexto
- selección de proveedor
- llamada al modelo
- quiz
- corrección local
- reintento de incorrectas

> Para la explicación detallada del bloque base, revisa el notebook original.
> Aquí nos enfocamos en la parte nueva: **OpenAI + Anthropic + Ollama + Quiz pro**.


In [3]:

# =========================================================
# Configuración base
# =========================================================

DEFAULT_TEMPERATURE = 0.3
DEFAULT_TOP_P = 0.9
DEFAULT_MAX_OUTPUT_TOKENS = 300
MAX_HISTORY_TURNS = 6

DEFAULT_OPENAI_MODEL = "gpt-4.1-mini"
DEFAULT_ANTHROPIC_MODEL = "claude-haiku-4-5"
DEFAULT_OLLAMA_MODEL = "llama3.2"

BASE_SYSTEM_PROMPT = """
Eres System Handler AI Study Assistant, un tutor experto, claro y paciente.

Tu misión es explicar cualquier tema de forma fácil de entender para cualquier persona,
incluyendo principiantes. Evita tecnicismos innecesarios.

Reglas:
1. Explica primero la idea principal en palabras simples.
2. Después da un ejemplo real o cotidiano.
3. Si ayuda, usa una analogía sencilla.
4. Cierra con un error común o una recomendación práctica.
5. Si el usuario pregunta por código, explica el código antes de mostrarlo.
6. No inventes datos. Si no sabes algo, dilo con honestidad.
""".strip()

QUIZ_AUTHOR_PROMPT = """
Vas a crear un mini quiz de 3 preguntas de opción múltiple sobre el tema pedido por el usuario.

Devuelve tu respuesta en DOS PARTES:

PARTE 1:
Un bloque JSON entre las etiquetas <quiz_json> y </quiz_json> con este formato exacto:
<quiz_json>
{
  "topic": "tema",
  "questions": [
    {
      "number": 1,
      "question": "texto",
      "options": {
        "a": "opción A",
        "b": "opción B",
        "c": "opción C"
      },
      "answer": "a",
      "explanation": "explicación breve"
    }
  ]
}
</quiz_json>

PARTE 2:
Después del JSON, escribe el quiz para el usuario en español, limpio y fácil de leer.

Reglas obligatorias:
- Exactamente 3 preguntas.
- Solo opciones a, b y c.
- Solo una respuesta correcta por pregunta.
- El tema debe ser el que pidió el usuario.
- Al final, pídele responder EXCLUSIVAMENTE con este formato:
  1c, 2a, 3b
- No muestres la clave correcta fuera del JSON.
""".strip()

QUIZ_ANSWER_PATTERN = re.compile(r"^\s*(\d+\s*[a-cA-C]\s*)(,\s*\d+\s*[a-cA-C]\s*)*$")


# =========================================================
# Utilidades de historial compatibles con Gradio 6+
# =========================================================

def normalize_chat_content(content: Any) -> str:
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    return str(content)


def append_history_message(history: Optional[List[Dict[str, Any]]], role: str, content: Any):
    if history is None:
        history = []
    safe_role = "assistant" if str(role) == "assistant" else "user"
    history.append({
        "role": safe_role,
        "content": normalize_chat_content(content)
    })
    return history


def history_to_chatbot_messages(history: Optional[List[Dict[str, Any]]]):
    safe_messages = []
    for item in history or []:
        if isinstance(item, dict):
            role = item.get("role", "assistant")
            content = item.get("content", "")
            if role not in {"user", "assistant"}:
                role = "assistant"
            safe_messages.append({
                "role": role,
                "content": normalize_chat_content(content)
            })
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            safe_messages.append({"role": "user", "content": normalize_chat_content(item[0])})
            safe_messages.append({"role": "assistant", "content": normalize_chat_content(item[1])})
    return safe_messages


def validate_chatbot_messages(messages, label="chatbot"):
    if not isinstance(messages, list):
        raise ValueError(f"[{label}] El chatbot debe recibir una lista.")
    for i, msg in enumerate(messages):
        if not isinstance(msg, dict):
            raise ValueError(f"[{label}] El mensaje #{i} no es dict: {msg}")
        if "role" not in msg or "content" not in msg:
            raise ValueError(f"[{label}] El mensaje #{i} no tiene role/content: {msg}")
        if msg["role"] not in {"user", "assistant"}:
            raise ValueError(f"[{label}] role inválido en #{i}: {msg}")
        msg["content"] = normalize_chat_content(msg["content"])
    return messages


# =========================================================
# Utilidades varias
# =========================================================

def provider_default_model(provider: str) -> str:
    if provider == "OpenAI":
        return DEFAULT_OPENAI_MODEL
    if provider == "Anthropic Claude":
        return DEFAULT_ANTHROPIC_MODEL
    return DEFAULT_OLLAMA_MODEL


def trim_history_for_model(history: Optional[List[Dict[str, Any]]], max_turns: int = MAX_HISTORY_TURNS):
    history = history or []
    return history[-(max_turns * 2):]


def build_user_prompt(user_message: str, history: Optional[List[Dict[str, Any]]], extra_context: str, study_mode: str) -> str:
    recent_history = trim_history_for_model(history)
    history_lines = []
    for item in recent_history:
        role = item.get("role", "assistant")
        content = normalize_chat_content(item.get("content", ""))
        history_lines.append(f"{role.upper()}: {content}")

    history_block = "\n".join(history_lines) if history_lines else "Sin historial previo."
    extra_context = (extra_context or "").strip() or "Sin contexto extra."

    return f"""
Modo de estudio: {study_mode}

Contexto extra:
{extra_context}

Historial reciente:
{history_block}

Mensaje actual del usuario:
{user_message}
""".strip()


def extract_quiz_json(text: str) -> Dict[str, Any]:
    match = re.search(r"<quiz_json>\s*(\{.*?\})\s*</quiz_json>", text, flags=re.DOTALL)
    if not match:
        raise ValueError("No se encontró el bloque <quiz_json>...</quiz_json>.")

    raw_json = match.group(1)
    data = json.loads(raw_json)

    if "topic" not in data or "questions" not in data:
        raise ValueError("El quiz no trae topic/questions.")

    questions = data["questions"]
    if not isinstance(questions, list) or len(questions) != 3:
        raise ValueError("El quiz debe tener exactamente 3 preguntas.")

    normalized_questions = []
    for i, q in enumerate(questions, start=1):
        options = q.get("options", {})
        answer = str(q.get("answer", "")).lower().strip()
        if set(options.keys()) != {"a", "b", "c"}:
            raise ValueError("Cada pregunta debe tener opciones a, b, c.")
        if answer not in {"a", "b", "c"}:
            raise ValueError("Cada pregunta debe tener answer válido: a, b o c.")

        normalized_questions.append({
            "number": int(q.get("number", i)),
            "question": q.get("question", ""),
            "options": {
                "a": options["a"],
                "b": options["b"],
                "c": options["c"],
            },
            "answer": answer,
            "explanation": q.get("explanation", "")
        })

    user_facing = re.sub(r"<quiz_json>\s*\{.*?\}\s*</quiz_json>", "", text, flags=re.DOTALL).strip()

    return {
        "topic": data["topic"],
        "questions": normalized_questions,
        "quiz_markdown": user_facing
    }


def parse_quiz_answers(answer_text: str) -> Dict[int, str]:
    if not QUIZ_ANSWER_PATTERN.match((answer_text or "").strip()):
        raise ValueError("Formato inválido. Usa algo como: 1c, 2a, 3b")

    answers = {}
    parts = [part.strip() for part in answer_text.split(",") if part.strip()]
    for part in parts:
        m = re.match(r"^(\d+)\s*([a-cA-C])$", part.replace(" ", ""))
        if not m:
            raise ValueError("Formato inválido. Usa algo como: 1c, 2a, 3b")
        number = int(m.group(1))
        option = m.group(2).lower()
        answers[number] = option
    return answers


def build_quiz_result_markdown(quiz_state: Dict[str, Any], user_answers: Dict[int, str]) -> Dict[str, Any]:
    rows = []
    correct_count = 0
    incorrect_questions = []
    reinforcement_topics = []

    for q in quiz_state["questions"]:
        number = q["number"]
        correct = q["answer"]
        user_answer = user_answers.get(number, "—")
        ok = user_answer == correct
        if ok:
            correct_count += 1
            result = "✅ Correcta"
        else:
            result = "❌ Incorrecta"
            incorrect_questions.append(q)
            reinforcement_topics.append(f"Pregunta {number}: {q['question']}")

        rows.append(
            f"| {number} | {q['question']} | {user_answer} | {correct} | {result} |"
        )

    total = len(quiz_state["questions"])
    percentage = round((correct_count / total) * 100, 2)

    feedback_lines = []
    for q in quiz_state["questions"]:
        number = q["number"]
        correct = q["answer"]
        user_answer = user_answers.get(number, "—")
        status = "bien" if user_answer == correct else "aquí fallaste"
        feedback_lines.append(
            f"**{number}.** Tu respuesta: `{user_answer}` · Correcta: `{correct}` · {status}.  \\n"
            f"Explicación: {q['explanation']}"
        )

    reinforcement_md = "\n".join([f"- {item}" for item in reinforcement_topics]) if reinforcement_topics else "- Ninguno. ¡Dominaste este quiz!"

    report = f"""
## Resultado del quiz

**Tema:** {quiz_state['topic']}  
**Aciertos:** {correct_count}/{total}  
**Porcentaje:** {percentage}%

### Tabla de resultados
| # | Pregunta | Tu respuesta | Correcta | Resultado |
|---|---|---|---|---|
{chr(10).join(rows)}

### Retroalimentación
{chr(10).join(feedback_lines)}

### Temas a reforzar
{reinforcement_md}
""".strip()

    retry_state = None
    if incorrect_questions:
        retry_state = {
            "topic": quiz_state["topic"],
            "questions": incorrect_questions
        }

    return {
        "report_markdown": report,
        "correct_count": correct_count,
        "total": total,
        "percentage": percentage,
        "retry_state": retry_state,
    }


def build_retry_quiz_intro_markdown(retry_state: Dict[str, Any]) -> str:
    lines = [
        f"## Reintento: {retry_state['topic']}",
        "",
        "Ahora responde solo las preguntas que fallaste.",
        "Usa este formato: `2a, 3b`",
        ""
    ]
    for q in retry_state["questions"]:
        lines.append(f"**{q['number']}. {q['question']}**")
        lines.append(f"a) {q['options']['a']}")
        lines.append(f"b) {q['options']['b']}")
        lines.append(f"c) {q['options']['c']}")
        lines.append("")
    return "\n".join(lines).strip()


def metrics_to_markdown(result: Dict[str, Any]) -> str:
    return f"""
### Métricas de la última respuesta
- **Proveedor:** {result['provider']}
- **Modelo:** {result['model_name']}
- **Mensajes enviados al modelo:** {result['messages_sent']}
- **Input tokens:** {result['input_tokens']}
- **Output tokens:** {result['output_tokens']}
- **Total tokens:** {result['total_tokens']}
- **Temperature usada:** {result['temperature_used']}
- **top_p usado:** {result['top_p_used']}
""".strip()


# =========================================================
# Llamadas a modelos
# =========================================================

def call_openai_model(system_prompt, user_prompt, model, temperature, top_p, max_tokens):
    if openai_client is None:
        raise ValueError("OpenAI no está configurado. Revisa OPENAI_API_KEY.")

    response = openai_client.responses.create(
        model=model,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=float(temperature),
        top_p=float(top_p),
        max_output_tokens=int(max_tokens),
    )
    return response.output_text


def call_anthropic_model(system_prompt, user_prompt, model, temperature, top_p, max_tokens):
    if anthropic_client is None:
        raise ValueError("Anthropic no está configurado. Revisa ANTHROPIC_API_KEY.")

    response = anthropic_client.messages.create(
        model=model,
        system=system_prompt,
        max_tokens=int(max_tokens),
        temperature=float(temperature),
        messages=[
            {"role": "user", "content": user_prompt}
        ],
    )

    chunks = []
    for block in response.content:
        if getattr(block, "type", None) == "text":
            chunks.append(block.text)
    return "\n".join(chunks).strip()


def call_ollama_model(system_prompt, user_prompt, model, temperature, top_p, max_tokens):
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "stream": False,
        "options": {
            "temperature": float(temperature),
            "top_p": float(top_p),
            "num_predict": int(max_tokens)
        }
    }

    response = requests.post(
        f"{OLLAMA_BASE_URL.rstrip('/')}/api/chat",
        json=payload,
        timeout=120
    )
    response.raise_for_status()
    data = response.json()
    return data.get("message", {}).get("content", "").strip()


def call_provider(provider, model, system_prompt, user_prompt, temperature, top_p, max_tokens):
    if provider == "OpenAI":
        return call_openai_model(system_prompt, user_prompt, model, temperature, top_p, max_tokens)
    if provider == "Anthropic Claude":
        return call_anthropic_model(system_prompt, user_prompt, model, temperature, top_p, max_tokens)
    if provider == "Ollama":
        return call_ollama_model(system_prompt, user_prompt, model, temperature, top_p, max_tokens)
    raise ValueError(f"Proveedor no soportado: {provider}")


def ask_model(provider, model_name, user_message, history, temperature, top_p, max_output_tokens, stable_mode, study_mode, extra_context):
    temperature_used = 0.0 if stable_mode else float(temperature)
    top_p_used = 1.0 if stable_mode else float(top_p)

    user_prompt = build_user_prompt(
        user_message=user_message,
        history=history,
        extra_context=extra_context,
        study_mode=study_mode,
    )

    answer = call_provider(
        provider=provider,
        model=model_name,
        system_prompt=BASE_SYSTEM_PROMPT,
        user_prompt=user_prompt,
        temperature=temperature_used,
        top_p=top_p_used,
        max_tokens=max_output_tokens,
    )

    input_tokens_est = max(1, len(user_prompt) // 4)
    output_tokens_est = max(1, len(answer) // 4)

    return {
        "provider": provider,
        "model_name": model_name,
        "answer": answer,
        "messages_sent": 2,
        "input_tokens": input_tokens_est,
        "output_tokens": output_tokens_est,
        "total_tokens": input_tokens_est + output_tokens_est,
        "temperature_used": temperature_used,
        "top_p_used": top_p_used,
    }


def generate_quiz(provider, model_name, topic, history, temperature, top_p, max_output_tokens, extra_context):
    prompt = f"Tema del quiz: {topic}\n\nGenera el quiz siguiendo exactamente las instrucciones."
    text = call_provider(
        provider=provider,
        model=model_name,
        system_prompt=QUIZ_AUTHOR_PROMPT,
        user_prompt=prompt,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_output_tokens,
    )
    parsed = extract_quiz_json(text)

    input_tokens_est = max(1, len(prompt) // 4)
    output_tokens_est = max(1, len(text) // 4)

    return {
        "quiz_state": parsed,
        "quiz_markdown": parsed["quiz_markdown"],
        "metrics": {
            "provider": provider,
            "model_name": model_name,
            "messages_sent": 2,
            "input_tokens": input_tokens_est,
            "output_tokens": output_tokens_est,
            "total_tokens": input_tokens_est + output_tokens_est,
            "temperature_used": temperature,
            "top_p_used": top_p,
        }
    }


# =========================================================
# Funciones del chat
# =========================================================

def update_model_by_provider(provider):
    return gr.update(value=provider_default_model(provider))


def retry_incorrect_fn(history, retry_state):
    history = history or []
    retry_state = retry_state or None

    if not retry_state:
        metrics = "### Métricas\nNo hay preguntas incorrectas pendientes por reintentar."
        chatbot_messages = validate_chatbot_messages(history_to_chatbot_messages(history), label="retry_empty")
        return chatbot_messages, history, None, retry_state, metrics, gr.update(interactive=False)

    intro = build_retry_quiz_intro_markdown(retry_state)
    append_history_message(history, "assistant", intro)

    metrics = "### Métricas\nModo reintento activado. Ahora responde solo las preguntas que fallaste."
    chatbot_messages = validate_chatbot_messages(history_to_chatbot_messages(history), label="retry_active")
    return chatbot_messages, history, retry_state, None, metrics, gr.update(interactive=False)


def clear_all():
    empty_chat = validate_chatbot_messages([], label="clear_all")
    return empty_chat, [], None, None, "### Métricas\nChat reiniciado.", "", gr.update(interactive=False)


def chat_fn(
    user_message,
    history,
    provider,
    model_name,
    study_mode,
    extra_context,
    temperature,
    top_p,
    max_output_tokens,
    stable_mode,
    quiz_state,
    retry_state,
):
    history = history or []
    quiz_state = quiz_state or None
    retry_state = retry_state or None
    user_message = (user_message or "").strip()

    if not user_message:
        chatbot_messages = validate_chatbot_messages(history_to_chatbot_messages(history), label="empty_input")
        return chatbot_messages, history, quiz_state, retry_state, "### Métricas\nNo escribiste ningún mensaje.", "", gr.update(interactive=bool(retry_state))

    append_history_message(history, "user", user_message)

    try:
        if study_mode == "Quiz":
            active_quiz = retry_state if retry_state else quiz_state

            if active_quiz is not None:
                try:
                    user_answers = parse_quiz_answers(user_message)
                except ValueError:
                    assistant_text = "Formato inválido. Responde así: `1c, 2a, 3b`. Si estás reintentando, por ejemplo: `2a, 3b`."
                    append_history_message(history, "assistant", assistant_text)
                    chatbot_messages = validate_chatbot_messages(history_to_chatbot_messages(history), label="quiz_invalid_format")
                    metrics = "### Métricas\nEsperando respuestas del quiz con formato válido."
                    return chatbot_messages, history, quiz_state, retry_state, metrics, "", gr.update(interactive=bool(retry_state))

                result = build_quiz_result_markdown(active_quiz, user_answers)
                append_history_message(history, "assistant", result["report_markdown"])

                if retry_state is not None:
                    quiz_state = None
                    retry_state = result["retry_state"]
                    metrics = "### Métricas\nSe corrigió el reintento de preguntas incorrectas."
                else:
                    quiz_state = None
                    retry_state = result["retry_state"]
                    metrics = "### Métricas\nSe corrigió el quiz completo."

                chatbot_messages = validate_chatbot_messages(history_to_chatbot_messages(history), label="quiz_corrected")
                return chatbot_messages, history, quiz_state, retry_state, metrics, "", gr.update(interactive=bool(retry_state))

            generated = generate_quiz(
                provider=provider,
                model_name=model_name,
                topic=user_message,
                history=history,
                temperature=float(temperature),
                top_p=float(top_p),
                max_output_tokens=int(max_output_tokens),
                extra_context=extra_context,
            )
            quiz_state = generated["quiz_state"]
            append_history_message(history, "assistant", generated["quiz_markdown"])
            chatbot_messages = validate_chatbot_messages(history_to_chatbot_messages(history), label="quiz_generated")
            metrics = metrics_to_markdown(generated["metrics"])
            return chatbot_messages, history, quiz_state, None, metrics, "", gr.update(interactive=False)

        result = ask_model(
            provider=provider,
            model_name=model_name,
            user_message=user_message,
            history=history[:-1],
            temperature=float(temperature),
            top_p=float(top_p),
            max_output_tokens=int(max_output_tokens),
            stable_mode=bool(stable_mode),
            study_mode=study_mode,
            extra_context=extra_context,
        )
        append_history_message(history, "assistant", result["answer"])
        chatbot_messages = validate_chatbot_messages(history_to_chatbot_messages(history), label="normal_chat")
        metrics = metrics_to_markdown(result)
        return chatbot_messages, history, None, None, metrics, "", gr.update(interactive=False)

    except Exception as e:
        append_history_message(history, "assistant", f"Ocurrió un error: {e}")
        chatbot_messages = validate_chatbot_messages(history_to_chatbot_messages(history), label="error_path")
        metrics = f"### Métricas\nError detectado: {e}"
        return chatbot_messages, history, quiz_state, retry_state, metrics, "", gr.update(interactive=bool(retry_state))



## Celda 6 — Demo rápida sin interfaz

Primero probamos la llamada con el proveedor que tú decidas.
Para OpenAI necesitas API key.
Para Anthropic necesitas API key.
Para Ollama necesitas tener el servidor local activo y el modelo descargado.


In [4]:
# Cambia este valor a "OpenAI", "Anthropic Claude" u "Ollama"
provider = "Ollama"

model_name = provider_default_model(provider)

# Descomenta para probar
# demo_result = ask_model(
#     provider=provider,
#     model_name=model_name,
#     user_message="Explícame qué es un token con un ejemplo cotidiano.",
#     history=[],
#     temperature=DEFAULT_TEMPERATURE,
#     top_p=DEFAULT_TOP_P,
#     max_output_tokens=DEFAULT_MAX_OUTPUT_TOKENS,
#     stable_mode=True,
#     study_mode="Explicar",
#     extra_context=""
# )
# print(demo_result["answer"])
# print("\nMétricas:")
# print(demo_result)



## Celda 7 — Reforzando el concepto de contexto

Aquí está una idea clave:

Si le preguntas a la IA **“¿cuál es mi nombre?”**, no lo sabe por arte de magia.

Solo lo sabrá si:
- ya apareció en el historial del chat, o
- tú lo agregaste explícitamente en el **contexto**.

Eso significa que el modelo **no tiene memoria personal automática**.
Trabaja con el contexto que le mandas en ese momento.


In [5]:
provider = "Ollama"
model_name = provider_default_model(provider)

# Descomenta para probar si Ollama está corriendo
# print("=== Caso 1: sin contexto extra ===")
# case_1 = ask_model(
#     provider=provider,
#     model_name=model_name,
#     user_message="¿Cuál es mi nombre?",
#     history=[],
#     temperature=0.1,
#     top_p=0.5,
#     max_output_tokens=120,
#     stable_mode=True,
#     study_mode="Explicar",
#     extra_context=""
# )
# print(case_1["answer"])
# print("\n" + "="*60 + "\n")
# print("=== Caso 2: con contexto extra ===")
# case_2 = ask_model(
#     provider=provider,
#     model_name=model_name,
#     user_message="¿Cuál es mi nombre?",
#     history=[],
#     temperature=0.1,
#     top_p=0.5,
#     max_output_tokens=120,
#     stable_mode=True,
#     study_mode="Explicar",
#     extra_context="Mi nombre es Rager."
# )
# print(case_2["answer"])



## Celda 8 — Modo Quiz pro

Ahora el quiz es más útil para estudiar:

- genera un mini quiz de 3 preguntas
- espera respuestas en este formato: `1c, 2a, 3b`
- corrige de forma **local y determinística**
- muestra tabla, porcentaje y retroalimentación
- activa un botón para **Reintentar incorrectas**



## Celda 9 — Interfaz visual

Aquí conectamos todo con Gradio.

En esta versión el `Chatbot` usa el formato de mensajes compatible con Gradio 6.x:

```python
{"role": "user", "content": "..."}
{"role": "assistant", "content": "..."}
```


In [6]:

with gr.Blocks() as demo:
    gr.Markdown(
        """
        # 🤖 System Handler – AI Study Assistant + Ollama
        Un asistente de estudio con tres caminos:
        - **OpenAI** para usar API en la nube
        - **Anthropic Claude** para probar otra familia de modelos
        - **Ollama** para correr modelos locales y practicar de forma **gratuita** en tu propia máquina

        ## Idea clave
        La IA solo sabe lo que le mandas en el contexto.
        """
    )

    history_state = gr.State(value=[])
    quiz_state_box = gr.State(value=None)
    retry_state_box = gr.State(value=None)

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label="Chat", height=500, value=[])

            user_input = gr.Textbox(
                label="Escribe tu mensaje o tus respuestas del quiz",
                placeholder="Ejemplo: Explícame qué es un microservicio... o en modo Quiz escribe solo el tema: microservicios. Para responder un quiz usa: 1c, 2a, 3b",
                lines=3
            )

            with gr.Row():
                send_btn = gr.Button("Enviar")
                clear_btn = gr.Button("Limpiar chat")
                retry_btn = gr.Button("Reintentar incorrectas", interactive=False)

        with gr.Column(scale=2):
            provider_dropdown = gr.Dropdown(
                choices=["OpenAI", "Anthropic Claude", "Ollama"],
                value="Ollama",
                label="Proveedor"
            )

            model_input = gr.Textbox(
                value=DEFAULT_OLLAMA_MODEL,
                label="Modelo",
                placeholder="Ejemplo: llama3.2, gpt-4.1-mini o claude-haiku-4-5"
            )

            study_mode_dropdown = gr.Dropdown(
                choices=["Explicar", "Quiz"],
                value="Explicar",
                label="Modo de estudio"
            )

            extra_context_box = gr.Textbox(
                label="Contexto extra (opcional)",
                placeholder="Ejemplo: Mi nombre es Rager y estoy estudiando microservicios.",
                lines=4
            )

            temperature_slider = gr.Slider(0.0, 1.5, value=DEFAULT_TEMPERATURE, step=0.1, label="Temperature")
            top_p_slider = gr.Slider(0.1, 1.0, value=DEFAULT_TOP_P, step=0.05, label="top_p")
            max_tokens_slider = gr.Slider(50, 1200, value=DEFAULT_MAX_OUTPUT_TOKENS, step=10, label="Max output tokens")
            stable_mode_checkbox = gr.Checkbox(value=False, label="Modo estable / más determinístico")

            metrics_box = gr.Markdown("### Métricas\nAún no hay interacciones.")

    provider_dropdown.change(
        fn=update_model_by_provider,
        inputs=[provider_dropdown],
        outputs=[model_input]
    )

    send_btn.click(
        fn=chat_fn,
        inputs=[
            user_input,
            history_state,
            provider_dropdown,
            model_input,
            study_mode_dropdown,
            extra_context_box,
            temperature_slider,
            top_p_slider,
            max_tokens_slider,
            stable_mode_checkbox,
            quiz_state_box,
            retry_state_box,
        ],
        outputs=[
            chatbot,
            history_state,
            quiz_state_box,
            retry_state_box,
            metrics_box,
            user_input,
            retry_btn,
        ]
    )

    user_input.submit(
        fn=chat_fn,
        inputs=[
            user_input,
            history_state,
            provider_dropdown,
            model_input,
            study_mode_dropdown,
            extra_context_box,
            temperature_slider,
            top_p_slider,
            max_tokens_slider,
            stable_mode_checkbox,
            quiz_state_box,
            retry_state_box,
        ],
        outputs=[
            chatbot,
            history_state,
            quiz_state_box,
            retry_state_box,
            metrics_box,
            user_input,
            retry_btn,
        ]
    )

    retry_btn.click(
        fn=retry_incorrect_fn,
        inputs=[history_state, retry_state_box],
        outputs=[chatbot, history_state, quiz_state_box, retry_state_box, metrics_box, retry_btn]
    )

    clear_btn.click(
        fn=clear_all,
        inputs=[],
        outputs=[chatbot, history_state, quiz_state_box, retry_state_box, metrics_box, user_input, retry_btn]
    )



## Celda 10 — Lanzar el asistente

Ejecuta esta celda para abrir la interfaz.


In [7]:
demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.



## Celda 11 — Ideas para probar

### OpenAI
- `Explícame qué es un prompt como si tuviera 10 años`
- `Dame un ejemplo simple de loops en Python`

### Anthropic Claude
- `Explícame la diferencia entre contexto y memoria`
- `Dame una analogía simple para entender embeddings`

### Ollama
- `Explícame qué es contexto`
- `Hazme un resumen breve de microservicios`

### Quiz
1. Cambia el modo a **Quiz**
2. Escribe en el chat el tema, por ejemplo:
   - `microservicios`
   - `java básico`
   - `python loops`
3. Responde el quiz con este formato exacto:

`1c, 2a, 3b`



## Celda 12 — Qué aprendiste en esta versión

Con este notebook ya practicaste varias ideas importantes:

- el mismo asistente puede conectarse a más de un proveedor
- ahora también soporta **Anthropic Claude**
- Ollama te deja experimentar localmente y de forma gratuita en tu máquina
- el contexto sigue siendo la clave para personalizar respuestas
- y el **modo Quiz** ya puede corregir respuestas con formato controlado: `1c, 2a, 3b`
- además, puedes **reintentar solo las incorrectas**

Si quieres profundizar en el código base del asistente, revisa el notebook anterior: `system-handler-ai-study-assistant`.
